# Custom Formula 1 neural network



In [ ]:
import os
import zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

## 1. Data loading


In [ ]:
DATA_DIR = "f1_data"
ZIP_URL = "https://github.com/TracingInsights/RaceData/releases/latest/download/data.zip"


def download_data():
    os.makedirs(DATA_DIR, exist_ok=True)
    if os.path.exists(os.path.join(DATA_DIR, "results.csv")):
        print("Data is already loaded")
        return

    print("Downloading data...")
    r = requests.get(ZIP_URL, timeout=60)
    r.raise_for_status()
    with open("f1_data.zip", "wb") as f:
        f.write(r.content)

    with zipfile.ZipFile("f1_data.zip", "r") as z:
        z.extractall(DATA_DIR)

    inner = os.path.join(DATA_DIR, "data")
    if os.path.exists(inner):
        for name in os.listdir(inner):
            os.replace(os.path.join(inner, name), os.path.join(DATA_DIR, name))
        os.rmdir(inner)


download_data()


def read_csv(name):
    return pd.read_csv(os.path.join(DATA_DIR, name))


races = read_csv("races.csv")
results = read_csv("results.csv")
drivers = read_csv("drivers.csv")
constructors = read_csv("constructors.csv")
circuits = read_csv("circuits.csv")
qualifying = read_csv("qualifying.csv")
status = read_csv("status.csv")

## 2. Dataset preparation


In [ ]:
drivers["driver"] = drivers["forename"] + " " + drivers["surname"]
constructors = constructors.rename(columns={"name": "constructor"})
circuits = circuits.rename(columns={"name": "circuit"})
races = races.rename(columns={"year": "season"})

qual_cols = qualifying[["raceId", "driverId", "position"]].rename(
    columns={"position": "qual_position"}
)

df = (
    results
    .merge(races[["raceId", "season", "round", "circuitId", "date"]], on="raceId")
    .merge(drivers[["driverId", "driver"]], on="driverId")
    .merge(constructors[["constructorId", "constructor"]], on="constructorId")
    .merge(circuits[["circuitId", "circuit", "country"]], on="circuitId")
    .merge(status[["statusId", "status"]], on="statusId", how="left")
    .merge(qual_cols, on=["raceId", "driverId"], how="left")
)

df = df.sort_values(["season", "round", "raceId", "driverId"]).reset_index(drop=True)

df["target_position"] = pd.to_numeric(df["positionOrder"], errors="coerce")
df["grid"] = pd.to_numeric(df["grid"], errors="coerce").replace(0, np.nan)
df["qual_position"] = pd.to_numeric(df["qual_position"], errors="coerce")
df["points"] = pd.to_numeric(df["points"], errors="coerce").fillna(0)

finished_mask = df["status"].fillna("").str.contains("Finished", case=False)
lap_mask = df["status"].fillna("").str.startswith("+")
df["dnf"] = (~(finished_mask | lap_mask)).astype(int)
df["is_podium"] = (df["target_position"] <= 3).astype(int)
df["weather"] = "unknown"

## 3. Feature engineering


In [ ]:
def add_features(data):
    data = data.copy()
    g_driver = data.groupby("driverId", group_keys=False)
    g_team = data.groupby("constructorId", group_keys=False)

    data["avg_position_last_3"] = g_driver["target_position"].apply(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )
    data["avg_position_season"] = data.groupby(["season", "driverId"], group_keys=False)[
        "target_position"
    ].apply(lambda x: x.shift(1).expanding().mean())
    data["podium_pct_season"] = data.groupby(["season", "driverId"], group_keys=False)[
        "is_podium"
    ].apply(lambda x: x.shift(1).expanding().mean())
    data["points_last_5"] = g_driver["points"].apply(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
    data["dnf_pct_last_5"] = g_driver["dnf"].apply(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
    data["constructor_avg_last_3"] = g_team["target_position"].apply(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )

    return data


df = add_features(df)

num_features = [
    "grid", "qual_position", "avg_position_last_3", "avg_position_season",
    "podium_pct_season", "points_last_5", "dnf_pct_last_5",
    "constructor_avg_last_3"
]
cat_features = ["driver", "constructor", "circuit", "country", "weather"]

for col in num_features:
    df[col] = df[col].fillna(df[col].median())

df = df[(df["season"] >= 2013) & (df["season"] <= 2023)].copy()
train = df[df["season"] <= 2022].copy()
test = df[df["season"] == 2023].copy()

print("Train:", train.shape)
print("Test 2023:", test.shape)

## 4. Data for the neural network


In [ ]:
X_all = pd.get_dummies(df[num_features + cat_features], columns=cat_features)
X_train = X_all.loc[train.index]
X_test = X_all.loc[test.index]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train = train["target_position"].values.astype("float32")
y_test = test["target_position"].values.astype("float32")

## 5. Custom neural network


In [ ]:
tf.random.set_seed(42)
np.random.seed(42)


class CustomF1NeuralNetwork(tf.keras.Model):
    """
    Custom neural network for predicting driver finishing position.
    It uses driver, constructor, and circuit features,
    and returns the expected race position.
    """

    def __init__(self):
        super().__init__()
        self.hidden_1 = tf.keras.layers.Dense(64, activation="relu")
        self.hidden_2 = tf.keras.layers.Dense(32, activation="relu")
        self.dropout = tf.keras.layers.Dropout(0.15)
        self.hidden_3 = tf.keras.layers.Dense(16, activation="relu")
        self.output_position = tf.keras.layers.Dense(1)

    def call(self, inputs, training=False):
        x = self.hidden_1(inputs)
        x = self.hidden_2(x)
        x = self.dropout(x, training=training)
        x = self.hidden_3(x)
        return self.output_position(x)


model = CustomF1NeuralNetwork()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=80,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

model.summary()

## 6. 2023 predictions


In [ ]:
test["predicted_position_raw"] = model.predict(X_test_scaled).reshape(-1)

# The model predicts a number, then drivers are ranked inside each race.
test["predicted_position"] = (
    test.groupby("raceId")["predicted_position_raw"]
    .rank(method="first")
    .astype(int)
)
test["error"] = test["predicted_position"] - test["target_position"]

mae = mean_absolute_error(test["target_position"], test["predicted_position"])
rmse = np.sqrt(mean_squared_error(test["target_position"], test["predicted_position"]))
r2 = r2_score(test["target_position"], test["predicted_position"])

metrics = pd.DataFrame([{
    "model": "Custom Neural Network",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
}])

print(metrics)

prediction_table = test[[
    "round", "driver", "constructor", "circuit",
    "grid", "qual_position", "target_position",
    "predicted_position", "predicted_position_raw", "error"
]].sort_values(["round", "predicted_position"])

prediction_table.head(30)

## 7. Season result


In [ ]:
F1_POINTS = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}

test["real_points_calc"] = test["target_position"].map(F1_POINTS).fillna(0)
test["pred_points_calc"] = test["predicted_position"].map(F1_POINTS).fillna(0)

season_compare = (
    test.groupby("driver")[["real_points_calc", "pred_points_calc"]]
    .sum()
    .reset_index()
)
season_compare["abs_error"] = (
    season_compare["real_points_calc"] - season_compare["pred_points_calc"]
).abs()
season_compare = season_compare.sort_values("real_points_calc", ascending=False)

season_compare

## 8. Charts


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="validation loss")
plt.title("Neural network training")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history["mae"], label="train MAE")
plt.plot(history.history["val_mae"], label="validation MAE")
plt.title("MAE during training")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()
plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(test["target_position"], test["predicted_position"], alpha=0.6)
plt.plot([1, 20], [1, 20], color="red")
plt.xlabel("Actual position")
plt.ylabel("Predicted position")
plt.title("Custom Neural Network: predicted vs actual")
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(test["error"], bins=20)
plt.title("Custom Neural Network: error distribution")
plt.xlabel("Position error")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(10, 5))
round_error = test.groupby("round")["error"].apply(lambda x: np.mean(np.abs(x)))
plt.plot(round_error.index, round_error.values, marker="o")
plt.title("Mean absolute error by 2023 race")
plt.xlabel("Race round")
plt.ylabel("MAE")
plt.grid(True)
plt.show()

top_season = season_compare.head(15)
plt.figure(figsize=(12, 5))
x = np.arange(len(top_season))
plt.bar(x - 0.2, top_season["real_points_calc"], width=0.4, label="Actual")
plt.bar(x + 0.2, top_season["pred_points_calc"], width=0.4, label="Predicted")
plt.xticks(x, top_season["driver"], rotation=60, ha="right")
plt.title("Custom Neural Network: 2023 actual and predicted points")
plt.ylabel("Points")
plt.legend()
plt.show()

for rnd in sorted(test["round"].unique())[:3]:
    print("\nRound", rnd)
    display(prediction_table[prediction_table["round"] == rnd][[
        "driver", "constructor", "target_position", "predicted_position", "error"
    ]])